<a href="https://colab.research.google.com/github/yunju-1118/ESAA/blob/OB/ESAA_OB2_project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **OB 2조 Project2**

https://www.kaggle.com/competitions/nlp-getting-started/overview

### [데이터 정보]

- id: 각 트윗(tweet)을 구분하기 위한 고유 식별자


- text: 트윗의 본문 내용(텍스트)


- location: 트윗이 발송된 위치 정보 (비어 있을 수도 있음)


- keyword: 트윗에서 추출된 특정 키워드 (비어 있을 수도 있음)


- target: train.csv에서만 존재하는 변수로, 트윗이 실제 재난에 관한 내용이면 1, 그렇지 않으면 0을 의미

In [ ]:
import pandas as pd


train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

In [ ]:
train.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
test.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [ ]:
print("[train shape]", train.shape)
print("[test  shape]", test.shape)
print("[cols]", list(train.columns))

[train shape] (7613, 5)
[test  shape] (3263, 4)
[cols] ['id', 'keyword', 'location', 'text', 'target']


Target 클래스 분포 확인

In [ ]:
print("\n[Target 분포 - 개수]")
print(train["target"].value_counts().rename_axis("target").to_frame("count"))

print("\n[Target 분포 - 비율]")
print(train["target"].value_counts(normalize=True).rename_axis("target").to_frame("ratio").round(5))


[Target 분포 - 개수]
        count
target       
0        4342
1        3271

[Target 분포 - 비율]
          ratio
target         
0       0.57034
1       0.42966


## **TF-IDF Vectorization 이용**

### **전처리**

정규식 기반(소문자/URL/멘션/해시태그/이모지/특수문자 제거) + 토큰화

In [ ]:
import nltk
for pkg in ("punkt", "punkt_tab"):
    nltk.download(pkg)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
import re
import nltk
nltk.download('punkt')

from nltk.tokenize import word_tokenize

_url   = re.compile(r'http\S+|www\.\S+')
_html  = re.compile(r'<.*?>')
_ment  = re.compile(r'@\w+')
_hash  = re.compile(r'#')
_emoji = re.compile("[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+", flags=re.UNICODE)
_nonalpha = re.compile(r"[^a-zA-Z\s]")

def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = _url.sub(" ", s)
    s = _html.sub(" ", s)
    s = _ment.sub(" ", s)
    s = _hash.sub("", s)
    s = _emoji.sub(" ", s)
    s = _nonalpha.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()

    tokens = word_tokenize(s)
    return " ".join(tokens)

train["text_clean"] = train["text"].fillna("").apply(clean_text)
test["text_clean"]  = test["text"].fillna("").apply(clean_text)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
train[["text", "text_clean"]]

,text,text_clean
0,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...
1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada
2,All residents asked to 'shelter in place' are ...,all residents asked to shelter in place are be...
3,"13,000 people receive #wildfires evacuation or...",people receive wildfires evacuation orders in ...
4,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...
...,...,...
7608,Two giant cranes holding a bridge collapse int...,two giant cranes holding a bridge collapse int...
7609,@aria_ahrary @TheTawniest The out of control w...,the out of control wild fires in california ev...
7610,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,m utc km s of volcano hawaii
7611,Police investigating after an e-bike collided ...,police investigating after an e bike collided ...


In [ ]:
test[["text", "text_clean"]]

,text,text_clean
0,Just happened a terrible car crash,just happened a terrible car crash
1,"Heard about #earthquake is different cities, s...",heard about earthquake is different cities sta...
2,"there is a forest fire at spot pond, geese are...",there is a forest fire at spot pond geese are ...
3,Apocalypse lighting. #Spokane #wildfires,apocalypse lighting spokane wildfires
4,Typhoon Soudelor kills 28 in China and Taiwan,typhoon soudelor kills in china and taiwan
...,...,...
3258,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...,earthquake safety los angeles safety fasteners...
3259,Storm in RI worse than last hurricane. My city...,storm in ri worse than last hurricane my city ...
3260,Green Line derailment in Chicago http://t.co/U...,green line derailment in chicago
3261,MEG issues Hazardous Weather Outlook (HWO) htt...,meg issues hazardous weather outlook hwo


학습/검증 분할

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_tr, X_va, y_tr, y_va = train_test_split(
    train["text_clean"], train["target"],
    test_size=0.2, random_state=42, stratify=train["target"]
)

# train+test 합쳐서 fit (사전 일치용!!)
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), max_df=0.9, min_df=2)
tfidf.fit(pd.concat([train["text_clean"], test["text_clean"]], axis=0))

Xtr = tfidf.transform(X_tr)
Xva = tfidf.transform(X_va)

In [ ]:
Xte = tfidf.transform(test["text_clean"])

TF-IDF 벡터화 (기본값으로 진행한 버전)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

clf = LogisticRegression(
    solver="liblinear",
    class_weight="balanced",
    random_state=42,
    max_iter=1000,
)
clf.fit(Xtr, y_tr)

va_pred = clf.predict(Xva)
print("ACC:", round(accuracy_score(y_va, va_pred), 4))
print("F1 :", round(f1_score(y_va, va_pred), 4))
print(classification_report(y_va, va_pred))

ACC: 0.7984
F1 : 0.7644
              precision    recall  f1-score   support

           0       0.82      0.83      0.82       869
           1       0.77      0.76      0.76       654

    accuracy                           0.80      1523
   macro avg       0.79      0.79      0.79      1523
weighted avg       0.80      0.80      0.80      1523



TfidfVectorizer의 하이퍼파라미터(ngram_range, max_df) 값을 바꿔가면서 예측 성능 비교

In [ ]:
param_list = [
    {"ngram_range": (1,1), "max_df": 1.0},
    {"ngram_range": (1,2), "max_df": 1.0},
    {"ngram_range": (1,2), "max_df": 0.95},
    {"ngram_range": (1,2), "max_df": 300},
    {"ngram_range": (1,3), "max_df": 0.9},
]

results = []

for params in param_list:
    print(f"\n ngram_range={params['ngram_range']}, max_df={params['max_df']}")

    tfidf = TfidfVectorizer(
        stop_words="english",
        ngram_range=params["ngram_range"],
        max_df=params["max_df"],
        max_features=20000
    )
    tfidf.fit(X_train)
    Xtr = tfidf.transform(X_train)
    Xva = tfidf.transform(X_valid)

    clf = LogisticRegression(solver="liblinear", class_weight="balanced", random_state=42)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xva)

    acc = accuracy_score(y_valid, pred)
    f1 = f1_score(y_valid, pred)
    results.append((params["ngram_range"], params["max_df"], acc, f1))

    print(f"  → Accuracy: {acc:.4f}, F1: {f1:.4f}")

res_df = pd.DataFrame(results, columns=["ngram_range", "max_df", "accuracy", "f1"])
display(res_df.sort_values("f1", ascending=False))



 ngram_range=(1, 1), max_df=1.0
  → Accuracy: 0.8155, F1: 0.7820

 ngram_range=(1, 2), max_df=1.0
  → Accuracy: 0.8155, F1: 0.7817

 ngram_range=(1, 2), max_df=0.95
  → Accuracy: 0.8155, F1: 0.7817

 ngram_range=(1, 2), max_df=300
  → Accuracy: 0.8155, F1: 0.7817

 ngram_range=(1, 3), max_df=0.9
  → Accuracy: 0.8221, F1: 0.7901


,ngram_range,max_df,accuracy,f1
4,"(1, 3)",0.90,0.822062,0.790085
0,"(1, 1)",1.00,0.815496,0.782002
1,"(1, 2)",1.00,0.815496,0.781663
2,"(1, 2)",0.95,0.815496,0.781663
3,"(1, 2)",300.00,0.815496,0.781663


가장 좋은 조합: ngram_range=(1, 3), max_df=0.90

### **모델링**

### **단일 모델**

#### Logistic Regression

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

In [ ]:
from sklearn.model_selection import GridSearchCV

lr = LogisticRegression(max_iter=1000, n_jobs=-1)

param_dist_lr = {
    "C": np.logspace(-4, 3, 15),
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"],
    "max_iter": [300, 500, 1000]
}

rs_lr = GridSearchCV(
    lr,
    param_grid=param_dist_lr,
    scoring="f1",
    cv=3,
    verbose=2,
    n_jobs=-1
)

rs_lr.fit(Xtr, y_tr)
print("Best Params (LogReg):", rs_lr.best_params_)

Fitting 3 folds for each of 180 candidates, totalling 540 fits
Best Params (LogReg): {'C': np.float64(3.1622776601683795), 'max_iter': 300, 'penalty': 'l2', 'solver': 'saga'}


Best Params (LogReg): {'C': np.float64(3.1622776601683795), 'max_iter': 300, 'penalty': 'l2', 'solver': 'saga'}

In [ ]:
model_log = LogisticRegression(**rs_lr.best_params_, n_jobs=-1)
model_log.fit(Xtr,y_tr)

y_pred_log = model_log.predict(Xva)

In [ ]:
print("######Evaluation(LR)###### \n")
acc = accuracy_score(y_va, y_pred_log)
f1 = f1_score(y_va, y_pred_log)
report = classification_report(y_va, y_pred_log)
print("Accuracy:",acc)
print("F1-score:", f1)
print("Classification report: \n", report)

######Evaluation(LR)###### 

Accuracy: 0.814182534471438
F1-score: 0.7712206952303962
Classification report: 
               precision    recall  f1-score   support

           0       0.81      0.88      0.84       869
           1       0.82      0.73      0.77       654

    accuracy                           0.81      1523
   macro avg       0.81      0.80      0.81      1523
weighted avg       0.81      0.81      0.81      1523



#### SVC

In [ ]:
svc = LinearSVC()

param_dist_svc = {
    "C": np.logspace(-4, 1, 10),
    "loss": ["hinge", "squared_hinge"],
    "max_iter": [1000, 3000, 5000]
}

rs_svc = GridSearchCV(
    svc,
    param_grid=param_dist_svc,
    scoring="f1",
    cv=3,
    verbose=2,
    n_jobs=-1
)

rs_svc.fit(Xtr, y_tr)
print("Best Params (SVM):", rs_svc.best_params_)

Fitting 3 folds for each of 60 candidates, totalling 180 fits
Best Params (SVM): {'C': np.float64(0.7742636826811278), 'loss': 'hinge', 'max_iter': 1000}


Best Params (SVM): {'C': np.float64(0.7742636826811278), 'loss': 'hinge', 'max_iter': 1000}

In [ ]:
model_svc = LinearSVC(**rs_svc.best_params_)
model_svc.fit(Xtr,y_tr)

y_pred_svc = model_svc.predict(Xva)

In [ ]:
print("######Evaluation(SVC)###### \n")
acc = accuracy_score(y_va, y_pred_svc)
f1 = f1_score(y_va, y_pred_svc)
report = classification_report(y_va, y_pred_svc)
print("Accuracy:",acc)
print("F1-score:", f1)
print("Classification report: \n", report)

######Evaluation(SVC)###### 

Accuracy: 0.8194353250164149
F1-score: 0.7706422018348624
Classification report: 
               precision    recall  f1-score   support

           0       0.80      0.90      0.85       869
           1       0.85      0.71      0.77       654

    accuracy                           0.82      1523
   macro avg       0.83      0.81      0.81      1523
weighted avg       0.82      0.82      0.82      1523



#### XGBoost

In [ ]:
xgb = XGBClassifier(
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False
)

param_dist_xgb = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": np.linspace(0.01, 0.3, 10),
    "subsample": np.linspace(0.6, 1.0, 5),
    "colsample_bytree": np.linspace(0.6, 1.0, 5)
}

rs_xgb = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist_xgb,
    n_iter=15,
    scoring="f1",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# XGBoost는 dense matrix 필요
rs_xgb.fit(Xtr.toarray(), y_tr)
print("Best Params (XGB):", rs_xgb.best_params_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [08:14:37] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Params (XGB): {'subsample': np.float64(1.0), 'n_estimators': 300, 'max_depth': 7, 'learning_rate': np.float64(0.3), 'colsample_bytree': np.float64(0.9)}


Best Params (XGB): {'subsample': np.float64(1.0), 'n_estimators': 300, 'max_depth': 7, 'learning_rate': np.float64(0.3), 'colsample_bytree': np.float64(0.9)}

In [ ]:
rs_xgb_params = {'subsample': np.float64(1.0), 'n_estimators': 300, 'max_depth': 7, 'learning_rate': np.float64(0.3), 'colsample_bytree': np.float64(0.9)}

In [ ]:
model_xgb = XGBClassifier(**rs_xgb_params)
model_xgb.fit(Xtr.toarray(),y_tr)

y_pred_xgb = model_xgb.predict(Xva.toarray())

In [ ]:
print("######Evaluation(XGBoost)###### \n")
acc = accuracy_score(y_va, y_pred_xgb)
f1 = f1_score(y_va, y_pred_xgb)
report = classification_report(y_va, y_pred_xgb)
print("Accuracy:",acc)
print("F1-score:", f1)
print("Classification report: \n", report)

######Evaluation(XGBoost)###### 

Accuracy: 0.799080761654629
F1-score: 0.7544141252006421
Classification report: 
               precision    recall  f1-score   support

           0       0.80      0.86      0.83       869
           1       0.79      0.72      0.75       654

    accuracy                           0.80      1523
   macro avg       0.80      0.79      0.79      1523
weighted avg       0.80      0.80      0.80      1523



#### RidgeClassifier

##### GridSearch

In [ ]:
from sklearn.linear_model import RidgeClassifier

In [ ]:
ridge = RidgeClassifier()
param_gird_ridge = {'alpha': [0.1,1,10,50,100]}

grid_ridge = GridSearchCV(
    ridge,
    param_grid = param_gird_ridge,
    scoring = "f1",
    cv = 5,
    verbose=2,
    n_jobs=-1
)

grid_ridge.fit(Xtr, y_tr)
print("Best Params (Ridge):", grid_ridge.best_params_)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best Params (Ridge): {'alpha': 1}


Best Params (Ridge): {'alpha': 1}

In [ ]:
model_ridge = RidgeClassifier(**grid_ridge.best_params_, class_weight="balanced")
model_ridge.fit(Xtr,y_tr)

y_pred_ridge = model_ridge.predict(Xva)

In [ ]:
print("######Evaluation(Ridge)###### \n")
acc = accuracy_score(y_va, y_pred_ridge)
f1 = f1_score(y_va, y_pred_ridge)
report = classification_report(y_va, y_pred_ridge)
print("Accuracy:",acc)
print("F1-score:", f1)
print("Classification report: \n", report)

######Evaluation(Ridge)###### 

Accuracy: 0.8030203545633617
F1-score: 0.7667185069984448
Classification report: 
               precision    recall  f1-score   support

           0       0.82      0.84      0.83       869
           1       0.78      0.75      0.77       654

    accuracy                           0.80      1523
   macro avg       0.80      0.80      0.80      1523
weighted avg       0.80      0.80      0.80      1523



#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import ComplementNB

In [ ]:
nb = ComplementNB()

param_grid_nb = {
    "alpha": np.linspace(0.001, 2, 20),
    "norm": [True, False]
}

grid_nb = GridSearchCV(
    nb,
    param_grid = param_grid_nb,
    cv = 5,
    scoring="f1",
    verbose=2,
    n_jobs=-1
)

grid_nb.fit(Xtr, y_tr)
print("Best Params(NB):", grid_nb.best_params_)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best Params(NB): {'alpha': np.float64(0.7374736842105264), 'norm': False}


Best Params(NB): {'alpha': np.float64(0.7374736842105264), 'norm': False

In [ ]:
model_nb = ComplementNB(**grid_nb.best_params_)
model_nb.fit(Xtr,y_tr)

y_pred_nb = model_nb.predict(Xva)

In [ ]:
print("######Evaluation(NB)###### \n")
acc = accuracy_score(y_va, y_pred_nb)
f1 = f1_score(y_va, y_pred_nb)
report = classification_report(y_va, y_pred_nb)
print("Accuracy:",acc)
print("F1-score:", f1)
print("Classification report: \n", report)

######Evaluation(NB)###### 

Accuracy: 0.8187787261982928
F1-score: 0.7688442211055276
Classification report: 
               precision    recall  f1-score   support

           0       0.80      0.91      0.85       869
           1       0.85      0.70      0.77       654

    accuracy                           0.82      1523
   macro avg       0.83      0.80      0.81      1523
weighted avg       0.82      0.82      0.82      1523



#### LightGBM

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
lgbm = LGBMClassifier()
param_grid_lgbm = {
    "num_leaves": [31, 63],
    "max_depth": [5, 7],
    "learning_rate": [0.05, 0.1, 0.2],
    "n_estimators": [300, 500],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [0, 0.5, 1.0]
}

grid_lgbm = GridSearchCV(
    lgbm,
    param_grid = param_grid_lgbm,
    cv = 3,
    scoring = "f1",
    verbose = 2,
    n_jobs=-1
)

grid_lgbm.fit(Xtr, y_tr)
print("Best Params(LGBM):", grid_lgbm.best_params_)

Fitting 3 folds for each of 288 candidates, totalling 864 fits
[LightGBM] [Info] Number of positive: 2617, number of negative: 3473
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.033451 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15841
[LightGBM] [Info] Number of data points in the train set: 6090, number of used features: 821
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.429721 -> initscore=-0.282990
[LightGBM] [Info] Start training from score -0.282990
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh

Best Params(LGBM): {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500, 'num_leaves': 31, 'reg_lambda': 1.0, 'subsample': 0.8}

In [ ]:
model_lgbm = LGBMClassifier(**grid_lgbm.best_params_)
model_lgbm.fit(Xtr,y_tr)

y_pred_lgbm = model_lgbm.predict(Xva)

[LightGBM] [Info] Number of positive: 2617, number of negative: 3473
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.223387 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15841
[LightGBM] [Info] Number of data points in the train set: 6090, number of used features: 821
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.429721 -> initscore=-0.282990
[LightGBM] [Info] Start training from score -0.282990
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
print("######Evaluation(LGBM)###### \n")
acc = accuracy_score(y_va, y_pred_lgbm)
f1 = f1_score(y_va, y_pred_lgbm)
report = classification_report(y_va, y_pred_lgbm)
print("Accuracy:",acc)
print("F1-score:", f1)
print("Classification report: \n", report)

######Evaluation(LGBM)###### 

Accuracy: 0.7951411687458962
F1-score: 0.7412935323383084
Classification report: 
               precision    recall  f1-score   support

           0       0.79      0.88      0.83       869
           1       0.81      0.68      0.74       654

    accuracy                           0.80      1523
   macro avg       0.80      0.78      0.79      1523
weighted avg       0.80      0.80      0.79      1523



### TruncatedSVD + LogisticRegression

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

In [ ]:
pipe_svd_lr = Pipeline([
    ('svd', TruncatedSVD(random_state=42)),
    ('lr', LogisticRegression(max_iter=1000, solver='liblinear'))
])

param_grid = {
    'svd__n_components': [50, 100, 150],
    'lr__C': np.logspace(-2, 1, 6),
    'lr__penalty': ['l1', 'l2']
}

# GridSearch
grid_svd_lr = GridSearchCV(
    pipe_svd_lr,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)


grid_svd_lr.fit(Xtr, y_tr)

print("Best params:", grid_svd_lr.best_params_)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params: {'lr__C': np.float64(10.0), 'lr__penalty': 'l1', 'svd__n_components': 150}


In [ ]:
model_svd_lr = grid_svd_lr.best_estimator_

y_pred_svd_lr = model_svd_lr.predict(Xva)

In [ ]:
print("######Evaluation(SVD+LR)###### \n")
acc = accuracy_score(y_va, y_pred_svd_lr)
f1 = f1_score(y_va, y_pred_svd_lr)
report = classification_report(y_va, y_pred_svd_lr)
print("Accuracy:",acc)
print("F1-score:", f1)
print("Classification report: \n", report)

######Evaluation(SVD+LR)###### 

Accuracy: 0.7655942219304005
F1-score: 0.703734439834025
Classification report: 
               precision    recall  f1-score   support

           0       0.76      0.85      0.81       869
           1       0.77      0.65      0.70       654

    accuracy                           0.77      1523
   macro avg       0.77      0.75      0.75      1523
weighted avg       0.77      0.77      0.76      1523



### TruncatedSVD + XGBoost

In [ ]:
pipe_svd_xgb = Pipeline([
    ('svd', TruncatedSVD(random_state=42)),
    ('xgb', XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1, use_label_encoder=False))
])

param_grid = {
    'svd__n_components': [50, 100, 150],
    "xgb__n_estimators": [200, 300, 400],
    "xgb__max_depth": [5, 7, 9],
    "xgb__learning_rate": np.linspace(0.01, 0.3, 10),
    "xgb__subsample": np.linspace(0.6, 1.0, 5),
    "xgb__colsample_bytree": np.linspace(0.6, 1.0, 5)
}

rs_svd_xgb = RandomizedSearchCV(pipe_svd_xgb,
                                param_distributions=param_grid,
                                n_iter=150, cv=3, scoring='f1_macro',
                                n_jobs=-1, verbose=2)

rs_svd_xgb.fit(Xtr, y_tr)

print("Best params:", rs_svd_xgb.best_params_)

Fitting 3 folds for each of 150 candidates, totalling 450 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:10:27] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best params: {'xgb__subsample': np.float64(0.8), 'xgb__n_estimators': 200, 'xgb__max_depth': 5, 'xgb__learning_rate': np.float64(0.07444444444444444), 'xgb__colsample_bytree': np.float64(0.6), 'svd__n_components': 150}


In [ ]:
model_svd_xgb = rs_svd_xgb.best_estimator_

y_pred_svd_xgb = model_svd_xgb.predict(Xva)

In [ ]:
print("######Evaluation(SVD+XGB)###### \n")
acc = accuracy_score(y_va, y_pred_svd_xgb)
f1 = f1_score(y_va, y_pred_svd_xgb)
report = classification_report(y_va, y_pred_svd_xgb)
print("Accuracy:",acc)
print("F1-score:", f1)
print("Classification report: \n", report)

######Evaluation(SVD+XGB)###### 

Accuracy: 0.7741300065659882
F1-score: 0.7084745762711865
Classification report: 
               precision    recall  f1-score   support

           0       0.76      0.88      0.82       869
           1       0.79      0.64      0.71       654

    accuracy                           0.77      1523
   macro avg       0.78      0.76      0.76      1523
weighted avg       0.78      0.77      0.77      1523



### **Voting & Stacking**

In [ ]:
import re, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier, StackingClassifier

SEED = 42
np.random.seed(SEED)

In [ ]:
X_all = train[["text_clean"]].copy()
y_all = train["target"].astype(int).copy()

X_tr, X_va, y_tr, y_va = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED)

#### 1) TF-IDF 파라미터(ngram_range, max_df, min_df 등)만 조정하며 미세 튜닝
 (모델은 Logistic 고정: C는 윤주가 찾은 최적값 사용했어용)

In [ ]:
# Logistic
LR_C = 43.162776601683795
logreg_fixed = LogisticRegression(
    C=LR_C, penalty="l2", solver="saga", max_iter=300, n_jobs=-1
)

# TF-IDF만 바꾸는 파이프라인
pipe_tfidf_lr = Pipeline([
    ("fe", ColumnTransformer(
        [("w", TfidfVectorizer(), "text_clean")],
        remainder="drop", n_jobs=-1
    )),
    ("lr", logreg_fixed)
])

param_dist_tfidf = {
    "fe__w__ngram_range": [(1,1), (1,2), (1,3)],
    "fe__w__min_df": [1, 2, 3, 5],
    "fe__w__max_df": [0.7, 0.8, 0.9, 0.95],
    "fe__w__max_features": [50000, 100000, 200000, None],
    "fe__w__sublinear_tf": [True, False],
    "fe__w__stop_words": [None, "english"],
    "fe__w__token_pattern": [r"(?u)\b\w+\b"],
    "fe__w__norm": ["l2"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rs = RandomizedSearchCV(
    estimator=pipe_tfidf_lr,
    param_distributions=param_dist_tfidf,
    n_iter=40,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
    random_state=SEED
)
rs.fit(X_tr, y_tr)

print("\n[Step1] Best TF-IDF params (CV F1=%.4f):" % rs.best_score_)
print(rs.best_params_)
pred_va = rs.predict(X_va)
print("[Step1] Hold-out ACC=%.4f  F1=%.4f" % (accuracy_score(y_va, pred_va), f1_score(y_va, pred_va)))
print(classification_report(y_va, pred_va))

Fitting 5 folds for each of 40 candidates, totalling 200 fits

[Step1] Best TF-IDF params (CV F1=0.7500):
{'fe__w__token_pattern': '(?u)\\b\\w+\\b', 'fe__w__sublinear_tf': False, 'fe__w__stop_words': None, 'fe__w__norm': 'l2', 'fe__w__ngram_range': (1, 2), 'fe__w__min_df': 1, 'fe__w__max_features': 50000, 'fe__w__max_df': 0.95}
[Step1] Hold-out ACC=0.7958  F1=0.7595
              precision    recall  f1-score   support

           0       0.82      0.83      0.82       869
           1       0.77      0.75      0.76       654

    accuracy                           0.80      1523
   macro avg       0.79      0.79      0.79      1523
weighted avg       0.80      0.80      0.80      1523



#### 2) 찾은 베스트 TF-IDF + Logistic 모델을 baseline으로 확정
   (best 파라미터로 전체 train 재학습)

In [ ]:
baseline_lr = rs.best_estimator_
# 전체 train으로 재학습
baseline_lr.fit(X_all, y_all)

# test 예측
proba_test_lr = baseline_lr.predict_proba(test[["text_clean"]])[:, 1]
pred_test_lr  = (proba_test_lr >= 0.5).astype(int)

#### 3) 같은 TF-IDF 벡터로 SVC, RidgeClf, NB 학습
 (Step1에서 얻은 TF-IDF 설정을 고정하여 재사용)

In [ ]:
# Best TF-IDF 벡터라이저 추출
best_vec: TfidfVectorizer = rs.best_estimator_.named_steps["fe"].transformers_[0][1]

# 동일 TF-IDF를 재사용하는 파이프라인들
pipe_svc = Pipeline([
    ("fe", ColumnTransformer([("w", best_vec, "text_clean")], remainder="drop", n_jobs=-1)),
    ("clf", CalibratedClassifierCV(LinearSVC(C=1.0), method="sigmoid", cv=5))
])

pipe_ridge = Pipeline([
    ("fe", ColumnTransformer([("w", best_vec, "text_clean")], remainder="drop", n_jobs=-1)),
    ("clf", CalibratedClassifierCV(RidgeClassifier(alpha=1.0), method="sigmoid", cv=5))
])

pipe_nb = Pipeline([
    ("fe", ColumnTransformer([("w", best_vec, "text_clean")], remainder="drop", n_jobs=-1)),
    ("clf", ComplementNB(alpha=0.5))
])

# Hold-out에서 점수 확인(가중치 산출 용)
models = {
    "log": baseline_lr,   # 이미 best_vec 내장
    "svc": pipe_svc,
    "rid": pipe_ridge,
    "cnb": pipe_nb
}

scores = {}
for name, mdl in models.items():
    # baseline_lr은 이미 fit(X_all,y_all) 되어 있으니, 공정 비교 위해 다시 X_tr로 재학습
    if name == "log":
        m = rs.best_estimator_
        m.fit(X_tr, y_tr)
    else:
        m = mdl.fit(X_tr, y_tr)

    if hasattr(m, "predict_proba"):
        p_va = m.predict_proba(X_va)[:,1]
        yhat = (p_va >= 0.5).astype(int)
    else:
        yhat = m.predict(X_va)

    scores[name] = (accuracy_score(y_va, yhat), f1_score(y_va, yhat))
    print(f"[Step3] {name}  ACC={scores[name][0]:.4f}  F1={scores[name][1]:.4f}")


[Step3] log  ACC=0.7965  F1=0.7604
[Step3] svc  ACC=0.8024  F1=0.7679
[Step3] rid  ACC=0.8037  F1=0.7684
[Step3] cnb  ACC=0.8102  F1=0.7610


| 모델                                      | Accuracy   | F1-Score   | 간단 해석                                                 |
| --------------------------------------- | ---------- | ---------- | ----------------------------------------------------- |
| **log** (Logistic Regression)           | 0.7965     | 0.7604     | baseline. 균형 좋은 기본 모델                                 |
| **svc** (Linear SVC + Calibrated)       | 0.8024     | 0.7679     | 약간 더 높음. 일반적으로 로지스틱보다 살짝 강한 마진 분류기                    |
| **rid** (Ridge Classifier + Calibrated) | 0.8037     | 0.7684     | 가장 안정적인 정확도. F1도 상위권                                  |
| **cnb** (Complement Naive Bayes)        | **0.8102** | **0.7610** | 정확도는 최고지만 F1은 살짝 낮음 (positive class recall 약간 낮을 가능성) |



지금 결과가 모두 0.76 ± 0.01 수준으로 비슷하므로,
→ Soft Voting 또는 Stacking 시 분류기 간 상관이 낮고 성능이 비슷한 이상적인 조합


#### 4) 이 4개를 보팅/스태킹에 활용
  - Soft Voting: 확률 평균(가중치=Hold-out F1)
  - Stacking: 메타모델로 로지스틱 사용

In [ ]:
# 4-1) Soft Voting (확률 출력 가능한 모델 구성)
voters = [
    ("log", rs.best_estimator_),
    ("svc", pipe_svc),
    ("rid", pipe_ridge),
    ("cnb", pipe_nb)
]

# 가중치 = Hold-out F1 비례
w = np.array([scores[k][1] for k,_ in voters])
w = w / w.sum()

vclf = VotingClassifier(estimators=voters, voting="soft", weights=w, n_jobs=-1)
vclf.fit(X_tr, y_tr)
p_va_v = vclf.predict_proba(X_va)[:,1]
yhat_v = (p_va_v >= 0.5).astype(int)
print("[Step4 - Voting] ACC=%.4f  F1=%.4f" % (accuracy_score(y_va, yhat_v), f1_score(y_va, yhat_v)))

# 4-2) Stacking (메타 모델: 로지스틱)
stack = StackingClassifier(
    estimators=voters,
    final_estimator=LogisticRegression(max_iter=1000),
    stack_method="predict_proba",  # base 모델들의 확률을 사용
    passthrough=False,
    cv=5, n_jobs=-1
)
stack.fit(X_tr, y_tr)
p_va_s = stack.predict_proba(X_va)[:,1]
yhat_s = (p_va_s >= 0.5).astype(int)
print("[Step4 - Stacking] ACC=%.4f  F1=%.4f" % (accuracy_score(y_va, yhat_s), f1_score(y_va, yhat_s)))

[Step4 - Voting] ACC=0.8076  F1=0.7709
[Step4 - Stacking] ACC=0.8135  F1=0.7742


- 두 앙상블 모두 개별 모델보다 개선됨

- Stacking이 Voting보다 살짝 더 우수 (F1 +0.0033, ACC +0.006)

→ 메타모델(Logistic)이 모델 간 가중 조합을 학습해 더 정교하게 판단함

## **Count 기반 Vectorization 이용**

### **전처리**

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

train_final = pd.read_csv('/content/drive/MyDrive/train.csv')
test_final = pd.read_csv('/content/drive/MyDrive/test.csv')
sub = pd.read_csv('/content/drive/MyDrive/sample_submission.csv')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re, html
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from nltk import pos_tag
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [ ]:
def nltk_pos_to_wn(pos):
    if pos.startswith('J'): return wordnet.ADJ
    if pos.startswith('V'): return wordnet.VERB
    if pos.startswith('N'): return wordnet.NOUN
    if pos.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

def clean_and_lemma_pos(text: str) -> str:
    text = html.unescape(str(text))
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s{2,}", " ", text).strip()
    toks = text.split()
    tagged = pos_tag(toks)
    return " ".join(lemmatizer.lemmatize(w, nltk_pos_to_wn(p)) for w,p in tagged)

train_final['clean_text_lemma2'] = train_final['text'].apply(clean_and_lemma_pos)
test_final['clean_text_lemma2'] = test_final['text'].apply(clean_and_lemma_pos)

In [ ]:
def normalize_repeats(text):
    # 문자 반복을 2번으로 제한: e.g., cooool → cool
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

train_final['clean_text_final'] = train_final['clean_text_lemma2'].apply(normalize_repeats)
test_final['clean_text_final'] = test_final['clean_text_lemma2'].apply(normalize_repeats)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text as sktext

domain_stop = {
    'aba',
    'abbswinston'
}
## aba, abbswinston이 사라지지 않아서 custom 버전으로 수정함

custom_stop = list(sktext.ENGLISH_STOP_WORDS.union(domain_stop))


count_vec_f = CountVectorizer(
    stop_words=custom_stop,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9,
    token_pattern=r'(?u)\b[a-z]{3,}\b'
)



In [ ]:
## 데이터셋 분리
from sklearn.model_selection import train_test_split

X = train_final['clean_text_final']
y = train_final['target']

X_train, X_val, y_train, y_val = train_test_split(X, y,
                                                    test_size=0.2, random_state=42, stratify=y)

In [ ]:
X_count = count_vec_f.fit_transform(train_final['clean_text_final'])

X_tr, X_v, y_tr, y_v = train_test_split(X_count, y, test_size=0.2, random_state=42, stratify=y)

### **모델링**

### **단일 모델**

#### Logistic Regression, SVC, XGBoost

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

In [ ]:
# SVM
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_tr, y_tr)
svm_pred = svm_model.predict(X_val)
svm_f1 = f1_score(y_val, svm_pred, average='macro')
print(f"🔹 SVM F1 (val): {svm_f1:.4f}")

# Logistic
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_tr, y_tr)
log_pred = log_model.predict(X_val)
log_f1 = f1_score(y_val, log_pred, average='macro')
print(f"🔹 Logistic Regression F1 (val): {log_f1:.4f}")

# XGBoost
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
xgb_model.fit(X_tr, y_tr)
xgb_pred = xgb_model.predict(X_val)
xgb_f1 = f1_score(y_val, xgb_pred, average='macro')
print(f"🔹 XGBoost F1 (val): {xgb_f1:.4f}")

🔹 SVM F1 (val): 0.7745
🔹 Logistic Regression F1 (val): 0.8033
🔹 XGBoost F1 (val): 0.7906


In [ ]:
scores = {
    'Logistic Regression': log_f1,
    'SVM': svm_f1,
    'XGBoost': xgb_f1
}
best_model_name = max(scores, key=scores.get)
print(f"\n✅ Best model on validation: {best_model_name} ({scores[best_model_name]:.4f})")


✅ Best model on validation: Logistic Regression (0.8033)


In [ ]:
best_model = {'Logistic Regression': log_model,
              'SVM': svm_model,
              'XGBoost': xgb_model}[best_model_name]

test_pred = best_model.predict(X_test)
print("\n📊 Classification report (Validation):")
print(classification_report(y_val, best_model.predict(X_val)))


📊 Classification report (Validation):
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       869
           1       0.81      0.72      0.77       654

    accuracy                           0.81      1523
   macro avg       0.81      0.80      0.80      1523
weighted avg       0.81      0.81      0.81      1523



##### GridSearch 적용

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grids = {
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'solver': ['liblinear', 'lbfgs']
    },
    'LinearSVC': {
        'C': [0.01, 0.1, 1, 10]
    },
    'XGBClassifier': {
        'n_estimators': [200, 300],
        'learning_rate': [0.05, 0.1],
        'max_depth': [4, 6],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }
}

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'LinearSVC': LinearSVC(random_state=42),
    'XGBClassifier': XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    )
}

# 결과 저장용 list
results = []

for name, model in models.items():
    print(f"\n🔍 {name} 튜닝 중...")

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        scoring='f1_macro',
        cv=3,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_tr, y_tr)

    best_model = grid.best_estimator_
    val_pred = best_model.predict(X_val)
    f1 = f1_score(y_val, val_pred, average='macro')

    results.append({
        'model': name,
        'best_params': grid.best_params_,
        'cv_best_f1': grid.best_score_,
        'val_f1': f1
    })

    print(f"✅ {name} 완료 | val F1={f1:.4f}")
    print(f"   Best params: {grid.best_params_}")


🔍 LogisticRegression 튜닝 중...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
✅ LogisticRegression 완료 | val F1=0.8027
   Best params: {'C': 1, 'solver': 'liblinear'}

🔍 LinearSVC 튜닝 중...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
✅ LinearSVC 완료 | val F1=0.8053
   Best params: {'C': 0.1}

🔍 XGBClassifier 튜닝 중...
Fitting 3 folds for each of 32 candidates, totalling 96 fits
✅ XGBClassifier 완료 | val F1=0.7906
   Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.8}


In [ ]:
# 결과 비교
results_df = pd.DataFrame(results).sort_values(by='val_f1', ascending=False)
print("\n📊 모델별 F1 비교 결과")
print(results_df.to_string(index=False))


📊 모델별 F1 비교 결과
             model                                                                                            best_params  cv_best_f1   val_f1
         LinearSVC                                                                                             {'C': 0.1}    0.778654 0.805299
LogisticRegression                                                                        {'C': 1, 'solver': 'liblinear'}    0.778971 0.802676
     XGBClassifier {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.8}    0.762327 0.790584


f1은 SVM이 가장 높음

In [ ]:
# 최고 성능 모델
best_row = results_df.iloc[0]
best_model_name = best_row['model']
best_model_params = best_row['best_params']

print(f"\n🏆 최종 선택 모델: {best_model_name}")
print("Best Params:", best_model_params)


🏆 최종 선택 모델: LinearSVC
Best Params: {'C': 0.1}


##### Optuna - logistic Regression


In [ ]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def objective_lr(trial):
    C = trial.suggest_loguniform('C', 1e-3, 1e3)
    penalty = trial.suggest_categorical('penalty', ['l2'])
    solver = 'lbfgs'

    model = LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=1000)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='macro')

# 실행
study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=30)
print("📘 Logistic Regression Best params:", study_lr.best_params)
print("📘 Logistic Regression Best F1:", study_lr.best_value)

[I 2025-11-12 03:07:10,700] A new study created in memory with name: no-name-4c14b3f8-d176-4385-96d4-132995ab2209
/tmp/ipython-input-2602592933.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('C', 1e-3, 1e3)
[I 2025-11-12 03:07:10,866] Trial 0 finished with value: 0.7484723369116433 and parameters: {'C': 118.64914371156848, 'penalty': 'l2'}. Best is trial 0 with value: 0.7484723369116433.
/tmp/ipython-input-2602592933.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform('C', 1e-3, 1e3)
[I 2025-11-12 03:07:10,900] Trial 1 finished with value: 0.7977865017033094 and parameters: {'C': 0.09335263532197773, 'penalt

📘 Logistic Regression Best params: {'C': 0.49155909806060233, 'penalty': 'l2'}
📘 Logistic Regression Best F1: 0.8083656888121944


##### Random Search - SVC

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from scipy.stats import loguniform
from sklearn.metrics import f1_score
import numpy as np

# 1단계: RandomizedSearchCV로 근사값 찾기
svm = SVC()

random_search = RandomizedSearchCV(
    estimator=svm,
    param_distributions={
        'C': loguniform(1e-2, 1e3),
        'gamma': loguniform(1e-4, 1e1),
        'kernel': ['rbf']
    },
    n_iter=30,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_tr, y_tr)

best_params_random = random_search.best_params_
print("1단계 최적 파라미터:", best_params_random)

# 2단계: 근처 범위로 GridSearchCV 재탐색
# 근처 범위를 자동으로 설정
C_center = best_params_random['C']
gamma_center = best_params_random['gamma']

# 로그스케일로 근처 탐색
C_grid = np.logspace(np.log10(C_center/3), np.log10(C_center*3), 5)
gamma_grid = np.logspace(np.log10(gamma_center/3), np.log10(gamma_center*3), 5)

param_grid = {
    'C': C_grid,
    'gamma': gamma_grid,
    'kernel': ['rbf']
}

grid_search = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_tr, y_tr)

best_params = grid_search.best_params_
best_f1_cv = grid_search.best_score_

# 검증 데이터셋 f1 계산
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_val)
f1_val = f1_score(y_val, y_pred)

print("\n✅ 최종 최적 파라미터:", best_params)
print("✅ 교차검증 평균 F1:", round(best_f1_cv, 4))
print("✅ 검증셋 F1:", round(f1_val, 4))

Fitting 3 folds for each of 30 candidates, totalling 90 fits
1단계 최적 파라미터: {'C': np.float64(145.28246637516014), 'gamma': np.float64(0.0011526449540315614), 'kernel': 'rbf'}
Fitting 3 folds for each of 25 candidates, totalling 75 fits

✅ 최종 최적 파라미터: {'C': np.float64(251.63661321069426), 'gamma': np.float64(0.00038421498467718703), 'kernel': 'rbf'}
✅ 교차검증 평균 F1: 0.7288
✅ 검증셋 F1: 0.7625


##### XGBoost 최적화

In [ ]:
from xgboost import XGBClassifier
import xgboost as xgb
from scipy import sparse
import numpy as np
from xgboost.callback import EarlyStopping

X_train_csr = sparse.csr_matrix(X_tr).astype(np.float32)
X_val_csr   = sparse.csr_matrix(X_v).astype(np.float32)
## 여기는 object가 들어가면 안 되고 벡터화가 끝난
## 데이터가 들어가야 함!!

dtrain = xgb.DMatrix(X_train_csr, label=y_train)
dval   = xgb.DMatrix(X_val_csr,   label=y_val)

pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
spw = neg / max(pos, 1)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "eta": 0.1,                 # learning_rate
    "max_depth": 6,
    "min_child_weight": 1.0,
    "gamma": 0.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "scale_pos_weight": spw,
    "tree_method": "hist",
    "seed": 42
}

watchlist = [(dtrain, "train"), (dval, "val")]
bst = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=2000,       # 충분히 크게 두고 조기종료로 컷
    evals=watchlist,
    early_stopping_rounds=50,
    verbose_eval=False
)

val_proba = bst.predict(dval, iteration_range=(0, bst.best_iteration + 1))

best_t, best_f1 = 0.5, -1.0
for t in np.linspace(0.2, 0.8, 61):
    pred_t = (val_proba >= t).astype(int)
    f1_t = f1_score(y_val, pred_t, average="macro")
    if f1_t > best_f1:
        best_f1, best_t = f1_t, t

print(f"Best threshold: {best_t:.3f} | F1(val): {best_f1:.4f}")
pred_final = (val_proba >= best_t).astype(int)
print(classification_report(y_val, pred_final))

Best threshold: 0.590 | F1(val): 0.8000
              precision    recall  f1-score   support

           0       0.79      0.90      0.84       869
           1       0.84      0.69      0.76       654

    accuracy                           0.81      1523
   macro avg       0.82      0.79      0.80      1523
weighted avg       0.81      0.81      0.81      1523



randomized search

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV, StratifiedKFold
from xgboost import XGBClassifier


xgb = XGBClassifier(
    objective='binary:logistic', tree_method='hist',
    eval_metric='logloss', n_jobs=-1, random_state=42
)

param_dist = {
    "learning_rate": np.logspace(-2.3, -0.7, 20),   # ~ [0.005, 0.2]
    "max_depth": np.arange(3, 10),
    "min_child_weight": np.logspace(-1, 1.3, 15),   # [0.1, 20]
    "gamma": np.linspace(0.0, 2.0, 11),
    "subsample": np.linspace(0.5, 0.9, 9),
    "colsample_bytree": np.linspace(0.5, 0.9, 9),
    "reg_alpha": np.logspace(-3, 0, 10),            # L1
    "reg_lambda": np.logspace(-2, 1, 12),           # L2
    "scale_pos_weight": [1.0, (y_train==0).sum()/max((y_train==1).sum(),1)]
}

cv = StratifiedKFold(5, shuffle=True, random_state=42)

halving = HalvingRandomSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    factor=3,                # 라운드마다 후보 1/3로 줄임
    resource='n_estimators', # 리소스 = 트리 개수
    max_resources=1200,      # 최종 라운드 트리 수
    min_resources=100,       # 초기 라운드 트리 수
    scoring='f1_macro',
    cv=cv,
    n_candidates=40,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
halving.fit(X_train_csr, y_train)

n_iterations: 3
n_required_iterations: 4
n_possible_iterations: 3
min_resources_: 100
max_resources_: 1200
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 40
n_resources: 100
Fitting 5 folds for each of 40 candidates, totalling 200 fits
----------
iter: 1
n_candidates: 14
n_resources: 300
Fitting 5 folds for each of 14 candidates, totalling 70 fits
----------
iter: 2
n_candidates: 5
n_resources: 900
Fitting 5 folds for each of 5 candidates, totalling 25 fits


HalvingRandomSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                      estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='logloss',
                                              feature_types=None,
                                              feature_weights=None, gamma=N...
       0.04641589, 0.1       , 0.21544347, 0.46415888, 1.        ]),
                                           'reg_lambda': array([ 0.01      ,  0.01873817,  0.03511192,  0.06579332,  0.12328467,
        0.23101297,  0.43287613,  0.81113083,  1.51991108,  2.84803587,
        5.33669923, 10.        ]),
                                           'scale_pos_weight': [1.0,
                                                                np.float64(1.327092090179595)],
                                           'subsample': array([0.5 , 0.55, 0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 ])},
                      random_state=42, resource='n_estimators',
                      scoring='f1_macro', verbose=1)

In [ ]:
from sklearn.metrics import f1_score, classification_report

y_val_pred = halving.best_estimator_.predict(X_val_csr)

f1_val = f1_score(y_val, y_val_pred, average='macro')
print(f"🔹 HalvingRandomSearchCV XGB F1 (val): {f1_val:.4f}")

print(classification_report(y_val, y_val_pred))

🔹 HalvingRandomSearchCV XGB F1 (val): 0.7959
              precision    recall  f1-score   support

           0       0.82      0.83      0.83       869
           1       0.77      0.76      0.77       654

    accuracy                           0.80      1523
   macro avg       0.80      0.80      0.80      1523
weighted avg       0.80      0.80      0.80      1523



Optuna

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 10.8 MB/s eta 0:00:00


In [ ]:
import optuna, numpy as np, xgboost as xgb
from sklearn.metrics import f1_score
from xgboost.callback import EarlyStopping

dtrain = xgb.DMatrix(X_train_csr, label=y_train)
dval   = xgb.DMatrix(X_val_csr,   label=y_val)
spw = (y_train==0).sum()/max((y_train==1).sum(),1)

def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "eta": trial.suggest_float("eta", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.1, 20, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "subsample": trial.suggest_float("subsample", 0.5, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
        "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1.0, float(spw)])
    }

    bst = xgb.train(
        params, dtrain,
        num_boost_round=2000,
        evals=[(dval, "val")],
        callbacks=[EarlyStopping(rounds=50, save_best=True)],
        verbose_eval=False
    )
    proba = bst.predict(dval, iteration_range=(0, bst.best_iteration+1))
    ts = np.linspace(0.2, 0.8, 19)
    f1s = [f1_score(y_val, (proba>=t).astype(int), average='macro') for t in ts]
    best = float(np.max(f1s))
    return best

study = optuna.create_study(direction='maximize', study_name='xgb_f1')
study.optimize(objective, n_trials=60, show_progress_bar=True)

print("Best params:", study.best_trial.params)
print("Best F1:", study.best_value)

[I 2025-11-11 04:32:42,826] A new study created in memory with name: xgb_f1


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2025-11-11 04:32:58,497] Trial 0 finished with value: 0.7946044501113929 and parameters: {'eta': 0.09470090301484403, 'max_depth': 3, 'min_child_weight': 0.20615284116233315, 'gamma': 0.5094380075846638, 'subsample': 0.8563873516753038, 'colsample_bytree': 0.6129347302240452, 'reg_alpha': 0.6238821581037635, 'reg_lambda': 6.131659166656368, 'scale_pos_weight': 1.0}. Best is trial 0 with value: 0.7946044501113929.
[I 2025-11-11 04:32:59,162] Trial 1 finished with value: 0.6703317453474962 and parameters: {'eta': 0.10235732809005879, 'max_depth': 4, 'min_child_weight': 18.37502081149067, 'gamma': 1.222509554985512, 'subsample': 0.8632657170017, 'colsample_bytree': 0.5073655364737807, 'reg_alpha': 0.012435924981873826, 'reg_lambda': 0.02334355220997676, 'scale_pos_weight': 1.327092090179595}. Best is trial 0 with value: 0.7946044501113929.
[I 2025-11-11 04:33:04,967] Trial 2 finished with value: 0.585126265581382 and parameters: {'eta': 0.011987711648825655, 'max_depth': 8, 'min_child_

#### 다항 나이브 베이즈

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report

pipe = make_pipeline(count_vec_f, MultinomialNB())

pipe.fit(X_train, y_train)
MNB_pred_val = pipe.predict(X_val)
MNB_f1 = f1_score(y_val, MNB_pred_val, average="macro")
print(f"MNB F1 (val) : {MNB_f1:.4f}")

MNB F1 (val) : 0.8003


In [ ]:
print(classification_report(y_val, MNB_pred_val))

              precision    recall  f1-score   support

           0       0.80      0.89      0.84       869
           1       0.83      0.70      0.76       654

    accuracy                           0.81      1523
   macro avg       0.81      0.80      0.80      1523
weighted avg       0.81      0.81      0.81      1523



In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'countvectorizer__ngram_range' : [(1,1), (1,2)],
    #'countvectorizer__min_df' : [2,5],
    #'countvectorizer__max_df' : [0.85, 0.9],
    #'countvectorizer__max_features' : [None, 30000, 50000],
    'multinomialnb__alpha' : [0.03, 0.1, 0.3, 0.7, 1.0, 1.5, 2.0, 3.0]
    }

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)

grid.fit(X_train, y_train)

print("Best params : ", grid.best_params_)
print("Best f1 : ", grid.best_score_)

Best params :  {'countvectorizer__ngram_range': (1, 2), 'multinomialnb__alpha': 1.0}
Best f1 :  0.7782812134441903


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'countvectorizer__ngram_range' : [(1,1), (1,2)],
    'countvectorizer__min_df' : [2,5],
    #'countvectorizer__max_df' : [0.85, 0.9],
    #'countvectorizer__max_features' : [None, 30000, 50000],
    'multinomialnb__alpha' : [0.03, 0.1, 0.3, 0.7, 1.0, 1.5, 2.0, 3.0]
    }

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)

grid.fit(X_train, y_train)

print("Best params : ", grid.best_params_)
print("Best f1 : ", grid.best_score_)

Best params :  {'countvectorizer__min_df': 2, 'countvectorizer__ngram_range': (1, 2), 'multinomialnb__alpha': 2.0}
Best f1 :  0.7847455928601439


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'countvectorizer__ngram_range' : [(1,1), (1,2)],
    'countvectorizer__min_df' : [2,5],
    'countvectorizer__max_df' : [0.85, 0.9],
    #'countvectorizer__max_features' : [None, 30000, 50000],
    'multinomialnb__alpha' : [0.03, 0.1, 0.3, 0.7, 1.0, 1.5, 2.0, 3.0]
    }

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)

grid.fit(X_train, y_train)

print("Best params : ", grid.best_params_)
print("Best f1 : ", grid.best_score_)

Best params :  {'countvectorizer__max_df': 0.85, 'countvectorizer__min_df': 2, 'countvectorizer__ngram_range': (1, 2), 'multinomialnb__alpha': 2.0}
Best f1 :  0.7847455928601439


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'countvectorizer__ngram_range' : [(1,1), (1,2)],
    'countvectorizer__min_df' : [2,5],
    'countvectorizer__max_df' : [0.85, 0.9],
    'countvectorizer__max_features' : [None, 30000, 50000],
    'multinomialnb__alpha' : [0.03, 0.1, 0.3, 0.7, 1.0, 1.5, 2.0, 3.0]
    }

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)

grid.fit(X_train, y_train)

print("Best params : ", grid.best_params_)
print("Best f1 : ", grid.best_score_)

Best params :  {'countvectorizer__max_df': 0.85, 'countvectorizer__max_features': None, 'countvectorizer__min_df': 2, 'countvectorizer__ngram_range': (1, 2), 'multinomialnb__alpha': 2.0}
Best f1 :  0.7847455928601439


나이브 베이즈는 GridSearch를 해도 성능 향상이 보이지는 않음..
기본 모델이 가장 성능이 좋음

### **Stacking**

#### 기본 모델

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier
import numpy as np


# 1) 베이스 모델
mnb = MultinomialNB()

lr_base = LogisticRegression(
    C=2.0,
    max_iter=2000,
    n_jobs=-1,
    solver="liblinear"
)

svm_base = LinearSVC(
    C=1.0
)

xgb_base = XGBClassifier(
  eta = 0.13498426500125352,
  max_depth = 9,
  min_child_weight = 0.233328646832991,
  gamma = 0.7319498152366601,
  subsample = 0.8117817790143049,
  colsample_bytree = 0.5234284051892206,
  reg_alpha = 0.004246704340785413,
  reg_lambda = 0.4763268757316294,
  scale_pos_weight = 1.327092090179595

)

estimators = [
    ("mnb", mnb),
    ("lr",  lr_base),
    ("svm", svm_base),
    ("xgb", xgb_base),
]

# 2) 메타 모델
final_lr = LogisticRegression(
    C=1.0,
    max_iter=2000,
    n_jobs=-1,
    solver="lbfgs"
)

stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=final_lr,
    cv=5,
    n_jobs=-1,
    stack_method="auto",
    passthrough=False
)

stack_clf.fit(X_tr, y_tr)

stack_pred_val = stack_clf.predict(X_v)
stack_f1 = f1_score(y_v, stack_pred_val, average="macro")
print(f"[Stacking] F1 (val, macro) : {stack_f1:.4f}")
print(classification_report(y_v, stack_pred_val))

[Stacking] F1 (val, macro) : 0.8193
              precision    recall  f1-score   support

           0       0.82      0.89      0.85       869
           1       0.84      0.74      0.78       654

    accuracy                           0.83      1523
   macro avg       0.83      0.81      0.82      1523
weighted avg       0.83      0.83      0.82      1523



In [ ]:
stack_proba_val = stack_clf.predict_proba(X_v)[:, 1]

ts = np.linspace(0.2, 0.8, 25)
f1s = [f1_score(y_v, (stack_proba_val >= t).astype(int), average="macro") for t in ts]

best_idx = int(np.argmax(f1s))
best_t = ts[best_idx]
print("Best threshold:", best_t)
print("Best F1 on val:", f1s[best_idx])

Best threshold: 0.5250000000000001
Best F1 on val: 0.8197499308113411


재학습 과정

In [ ]:
X_full = count_vec_f.fit_transform(train_final['clean_text_final'])
y_full = train_final['target']

X_test = count_vec_f.transform(test_final['clean_text_final'])

stack_clf_full = StackingClassifier(
    estimators=estimators,
    final_estimator=final_lr,
    cv=5,
    n_jobs=-1,
    stack_method="auto",
    passthrough=False
)

stack_clf_full.fit(X_full, y_full)

test_proba = stack_clf_full.predict_proba(X_test)[:, 1]

threshold = best_t
test_pred = (test_proba >= threshold).astype(int)

sub['target'] = test_pred
sub.to_csv('/content/drive/MyDrive/stacking_submission.csv', index=False)


In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/stacking_submission.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

기본 스태킹 모델 : 리더보드 점수 0.79282

#### type 2


In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import f1_score, classification_report

from xgboost import XGBClassifier
from google.colab import files

y_full = train_final['target'].values

X_full = count_vec_f.fit_transform(train_final['clean_text_final'])
X_test = count_vec_f.transform(test_final['clean_text_final'])

X_tr, X_v, y_tr, y_v = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

print("X_tr shape:", X_tr.shape, "X_v shape:", X_v.shape)


X_tr shape: (6090, 2980) X_v shape: (1523, 2980)


In [ ]:
# 1. 베이스 모델 정의

# 1) Multinomial Naive Bayes
mnb = MultinomialNB(alpha = 0.3)

# 2) Logistic Regression
lr_base = LogisticRegression(
    C=0.49155909806060233,
    penalty='l2',
    solver='liblinear',
    max_iter=3000,
    n_jobs=-1
)

# 3) SVC
svc_base = SVC(
    C=251.63661321069426,
    gamma=0.00038421498467718703,
    kernel='rbf',
    probability=True,
    random_state=42
)

xgb_base = XGBClassifier(
  eta = 0.13498426500125352,
  max_depth = 9,
  min_child_weight = 0.233328646832991,
  gamma = 0.7319498152366601,
  subsample = 0.8117817790143049,
  colsample_bytree = 0.5234284051892206,
  reg_alpha = 0.004246704340785413,
  reg_lambda = 0.4763268757316294,
  scale_pos_weight = 1.327092090179595
)

estimators = [
    ("mnb", mnb),
    ("lr",  lr_base),
    ("svc", svc_base),
    ("xgb", xgb_base),
]

# --------------------------------------------
# 2. 메타 모델
# --------------------------------------------
final_lr = LogisticRegression(
    C=1.0,
    penalty='l2',
    solver='lbfgs',
    max_iter=3000,
    n_jobs=-1
)

stack_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=final_lr,
    cv=5,
    n_jobs=-1,
    stack_method="auto",
    passthrough=False
)

# 3. train/validation 기준 학습 + 성능 확인
stack_clf.fit(X_tr, y_tr)

val_pred = stack_clf.predict(X_v)
val_f1 = f1_score(y_v, val_pred, average='macro')
print(f"[Stacking] F1-macro (val, threshold=0.5): {val_f1:.4f}")
print(classification_report(y_v, val_pred))

[Stacking] F1-macro (val, threshold=0.5): 0.8218
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       869
           1       0.84      0.74      0.79       654

    accuracy                           0.83      1523
   macro avg       0.83      0.82      0.82      1523
weighted avg       0.83      0.83      0.83      1523



In [ ]:
val_proba = stack_clf.predict_proba(X_v)[:, 1]

ts = np.linspace(0.2, 0.8, 25)
f1s = [f1_score(y_v, (val_proba >= t).astype(int), average="macro") for t in ts]

best_idx = int(np.argmax(f1s))
best_t2 = ts[best_idx]
print("Best threshold on val:", best_t)
print("Best F1 on val:", f1s[best_idx])

Best threshold on val: 0.4750000000000001
Best F1 on val: 0.8218054668325321


In [ ]:
stack_clf_full = StackingClassifier(
    estimators=estimators,
    final_estimator=final_lr,
    cv=5,
    n_jobs=-1,
    stack_method="auto",
    passthrough=False
)

stack_clf_full.fit(X_full, y_full)

StackingClassifier(cv=5,
                   estimators=[('mnb', MultinomialNB()),
                               ('lr',
                                LogisticRegression(C=0.49155909806060233,
                                                   max_iter=3000, n_jobs=-1,
                                                   solver='liblinear')),
                               ('svc',
                                SVC(C=251.63661321069426,
                                    gamma=0.00038421498467718703,
                                    probability=True, random_state=42)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_byno...
                                              interaction_constraints=None,
                                              learning_rate=None, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=9,
                                              max_leaves=None,
                                              min_child_weight=0.233328646832991,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=None, n_jobs=None, ...))],
                   final_estimator=LogisticRegression(max_iter=3000, n_jobs=-1),
                   n_jobs=-1)

In [ ]:
test_proba = stack_clf_full.predict_proba(X_test)[:, 1]

threshold = best_t2 if 'best_t' in globals() else 0.5
print("Final threshold used:", threshold)

test_pred = (test_proba >= threshold).astype(int)

sub = pd.read_csv("/content/drive/MyDrive/sample_submission.csv")
sub['target'] = test_pred

save_path = "/content/stacking_submission_optuna_svc.csv"
sub.to_csv(save_path, index=False)

files.download(save_path)

Final threshold used: 0.5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2번째 스태킹 모델 리더보드 점수 0.79650

#### 메타 모델 변경

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import f1_score, classification_report

from xgboost import XGBClassifier
from scipy.sparse import hstack
from google.colab import files

y_full = train_final['target'].values

word_vec = CountVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=3,
    analyzer='word'
)

char_vec = CountVectorizer(
    ngram_range=(3, 5),
    min_df=3,
    analyzer='char'
)
X_word = word_vec.fit_transform(train_final['clean_text_final'])
X_char = char_vec.fit_transform(train_final['clean_text_final'])
X_full = hstack([X_word, X_char])


X_test_word = word_vec.transform(test_final['clean_text_final'])
X_test_char = char_vec.transform(test_final['clean_text_final'])
X_test = hstack([X_test_word, X_test_char])

X_tr, X_v, y_tr, y_v = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

print("X_tr shape:", X_tr.shape, "X_v shape:", X_v.shape)



X_tr shape: (6090, 62717) X_v shape: (1523, 62717)


In [ ]:
# 1) 메타 모델을 XGBoost

estimators = [
    ("mnb", mnb),
    ("lr",  lr_base),
    ("svc", svc_base),
    ("xgb", xgb_base),
]


final_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)


stack_clf_xgb = StackingClassifier(
    estimators=estimators,
    final_estimator=final_xgb,
    cv=5,
    n_jobs=-1,
    stack_method="auto",
    passthrough=False
)

stack_clf_xgb.fit(X_tr, y_tr)

val_pred = stack_clf_xgb.predict(X_v)
val_f1 = f1_score(y_v, val_pred, average='macro')
print(f"\n[Validation] F1-macro (threshold=0.5): {val_f1:.4f}")
print(classification_report(y_v, val_pred))


[Validation] F1-macro (threshold=0.5): 0.8063
              precision    recall  f1-score   support

           0       0.80      0.90      0.85       869
           1       0.84      0.70      0.77       654

    accuracy                           0.81      1523
   macro avg       0.82      0.80      0.81      1523
weighted avg       0.82      0.81      0.81      1523



In [ ]:
val_proba = stack_clf_xgb.predict_proba(X_v)[:, 1]


ts = np.linspace(0.2, 0.8, 25)
f1s = [f1_score(y_v, (val_proba >= t).astype(int), average="macro") for t in ts]

best_idx = int(np.argmax(f1s))
best_t_xgb = ts[best_idx]
print(f"\n[Validation] Best threshold: {best_t_xgb:.4f}")
print(f"[Validation] Best F1-macro: {f1s[best_idx]:.4f}")


[Validation] Best threshold: 0.4750
[Validation] Best F1-macro: 0.8128


In [ ]:
stack_clf_xgb_full = StackingClassifier(
    estimators=estimators,
    final_estimator=final_xgb,
    cv=5,
    n_jobs=-1,
    stack_method="auto",
    passthrough=False
)

stack_clf_xgb_full.fit(X_full, y_full)

StackingClassifier(cv=5,
                   estimators=[('mnb', MultinomialNB()),
                               ('lr',
                                LogisticRegression(C=0.49155909806060233,
                                                   max_iter=3000, n_jobs=-1,
                                                   solver='liblinear')),
                               ('svc',
                                SVC(C=251.63661321069426,
                                    gamma=0.00038421498467718703,
                                    probability=True, random_state=42)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_byno...
                                                 feature_weights=None,
                                                 gamma=None, grow_policy=None,
                                                 importance_type=None,
                                                 interaction_constraints=None,
                                                 learning_rate=0.05,
                                                 max_bin=None,
                                                 max_cat_threshold=None,
                                                 max_cat_to_onehot=None,
                                                 max_delta_step=None,
                                                 max_depth=3, max_leaves=None,
                                                 min_child_weight=None,
                                                 missing=nan,
                                                 monotone_constraints=None,
                                                 multi_strategy=None,
                                                 n_estimators=300, n_jobs=-1,
                                                 num_parallel_tree=None, ...),
                   n_jobs=-1)

In [ ]:
test_proba = stack_clf_xgb_full.predict_proba(X_test)[:, 1]

threshold = best_t_xgb
print("Final threshold used:", threshold)

test_pred = (test_proba >= threshold).astype(int)

Final threshold used: 0.4750000000000001


In [ ]:
sub = pd.read_csv("/content/drive/MyDrive/sample_submission.csv")
sub["target"] = test_pred

save_path = "/content/stacking_metaXGB_word_char_count_submission.csv"
sub.to_csv(save_path, index=False)

files.download(save_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

마지막 스태킹 모델 리더보드 점수 : 0.79742

>> 이 모델이 스태킹 모델에서는 가장 성능이 좋음

### **Voting**

#### average = macro

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score, classification_report

# Logistic Regression
log_reg = LogisticRegression(
    C=0.49155909806060233,
    penalty='l2',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

# SVM
svm_clf = SVC(
    C=251.63661321069426,
    gamma=0.00038421498467718703,
    kernel='rbf',
    probability=True,   # 보팅(soft voting) 위해 필요
    random_state=42
)

# MultinomialNB
mnb_clf = MultinomialNB()

# XGBoost
xgb_clf = XGBClassifier(
    eta=0.13498426500125352,
    max_depth=9,
    min_child_weight=0.233328646832991,
    gamma=0.7319498152366601,
    subsample=0.8117817790143049,
    colsample_bytree=0.5234284051892206,
    reg_alpha=0.004246704340785413,
    reg_lambda=0.4763268757316294,
    scale_pos_weight=1.327092090179595,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# voting_clf
voting_clf = VotingClassifier(
    estimators=[
        ('logreg', log_reg),
        ('svm', svm_clf),
        ('mnb', mnb_clf),
        ('xgb', xgb_clf)
    ],
    voting='soft'   # 확률 기반 보팅
)

# 학습
voting_clf.fit(X_train, y_train)

# 검증 예측
voting_pred = voting_clf.predict(X_val)

# 평가
voting_f1 = f1_score(y_val, voting_pred, average='macro')
print(f"Voting Classifier F1 (val): {voting_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, voting_pred))
print(f"✅ Voting Classifier F1 Score (val): {voting_f1:.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [00:42:28] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Voting Classifier F1 (val): 0.8682

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.95      0.90       869
           1       0.92      0.78      0.84       654

    accuracy                           0.87      1523
   macro avg       0.88      0.86      0.87      1523
weighted avg       0.88      0.87      0.87      1523

✅ Voting Classifier F1 Score (val): 0.8682


#### average = binomial

In [ ]:
# 개별 모델 정의
log_reg = LogisticRegression(
    C=0.49155909806060233,
    penalty='l2',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

svm_clf = SVC(
    C=251.63661321069426,
    gamma=0.00038421498467718703,
    kernel='rbf',
    probability=True,
    random_state=42
)

mnb_clf = MultinomialNB()

xgb_clf = XGBClassifier(
    eta=0.13498426500125352,
    max_depth=9,
    min_child_weight=0.233328646832991,
    gamma=0.7319498152366601,
    subsample=0.8117817790143049,
    colsample_bytree=0.5234284051892206,
    reg_alpha=0.004246704340785413,
    reg_lambda=0.4763268757316294,
    scale_pos_weight=1.327092090179595,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# voting_clf
voting_clf = VotingClassifier(
    estimators=[
        ('logreg', log_reg),
        ('svm', svm_clf),
        ('mnb', mnb_clf),
        ('xgb', xgb_clf)
    ],
    voting='soft'
)

voting_clf.fit(X_train, y_train)

voting_pred = voting_clf.predict(X_val)

voting_f1_binary = f1_score(y_val, voting_pred, average="binary")
print(f"✅ Voting Classifier F1 Score (binary, val): {voting_f1_binary:.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [00:43:54] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Voting Classifier F1 Score (binary, val): 0.8408


-> average = macro가 더 나음

#### 최종 모델

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score

# 개별 모델 정의
log_reg = LogisticRegression(
    C=0.49155909806060233,
    penalty='l2',
    solver='lbfgs',
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

svm_clf = SVC(
    C=251.63661321069426,
    gamma=0.00038421498467718703,
    kernel='rbf',
    probability=True,
    random_state=42,
    class_weight='balanced'
)

mnb_clf = MultinomialNB()

xgb_clf = XGBClassifier(
    eta=0.13498426500125352,
    max_depth=9,
    min_child_weight=0.233328646832991,
    gamma=0.7319498152366601,
    subsample=0.8117817790143049,
    colsample_bytree=0.5234284051892206,
    reg_alpha=0.004246704340785413,
    reg_lambda=0.4763268757316294,
    scale_pos_weight=1.327092090179595,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# f1 score 기반 가중치 조절
weights = [0.8084, 0.7625, 0.70, 0.8111]


# voting_clf
voting_clf = VotingClassifier(
    estimators=[
        ('logreg', log_reg),
        ('svm', svm_clf),
        ('mnb', mnb_clf),
        ('xgb', xgb_clf)
    ],
    voting='soft',
    weights=weights
)

# 모델 학습
voting_clf.fit(X_train, y_train)
val_pred = voting_clf.predict(X_val)

f1_macro = f1_score(y_val, val_pred, average='macro')
print(f"✅ VotingClassifier (macro F1): {f1_macro:.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [00:56:26] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ VotingClassifier (macro F1): 0.8746


Count 벡터 기반에서는 MultinomialNB의 역할이 좀 더 커짐.
Naive Bayes는 TF-IDF보다 Count 기반에서 훨씬 잘 작동하니까
가중치를 약간 더 높여주는 게 좋음.

이렇다 해서 조정했는데 성능은 조금 떨어짐.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score

# 개별 모델 정의
log_reg = LogisticRegression(C=0.35, penalty='l2', class_weight='balanced', max_iter=1000, random_state=42)
svm_clf = SVC(C=251.6366, gamma=0.0003842, kernel='rbf', probability=True, class_weight='balanced', random_state=42)
mnb_clf = MultinomialNB()
xgb_clf = XGBClassifier(
    eta=0.08, max_depth=7, gamma=0.4, subsample=0.9, colsample_bytree=0.7,
    reg_alpha=0.01, reg_lambda=0.8, scale_pos_weight=1.2, eval_metric='logloss',
    use_label_encoder=False, random_state=42
)

# 가중치 조정 (MNB ↑)
weights = [0.80, 0.76, 0.78, 0.81]

voting_clf = VotingClassifier(
    estimators=[
        ('logreg', log_reg),
        ('svm', svm_clf),
        ('mnb', mnb_clf),
        ('xgb', xgb_clf)
    ],
    voting='soft',
    weights=weights
)

# 학습 및 검증
voting_clf.fit(X_train, y_train)
pred_val = voting_clf.predict(X_val)
f1_macro = f1_score(y_val, pred_val, average='macro')
print(f"✅ VotingClassifier (macro F1): {f1_macro:.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [00:51:00] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ VotingClassifier (macro F1): 0.8611
